# 📐 Curve Fitting Sandbox
This notebook provides a decoupled environment for testing and debugging swap curve fitting algorithms using static market data.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time
from dataclasses import dataclass, field
import sys
import os

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Internal Imports
from pricing.marketmodels.integrated_rate_curve import IntegratedShortRateCurve, IntegratedRatePoint
from pricing.marketmodels.curve_fitter import CurveFitter
from pricing.marketmodels.yield_curve import YieldCurvePoint, LinearTermDiscountCurve
from pricing.instruments.ir_swap_fixed_floatapprox import IRSwapFixedFloatApprox
from pricing.marketmodels.swap_curve import SwapQuote

### 📥 Input Market Quotes
Set the par swap rates (in decimals) for the USD OIS curve.

In [ ]:
# Static Quote Data (Modify these values to change the curve)
market_quotes = {
    "1Y": 0.0450,
    "2Y": 0.0420,
    "3Y": 0.0405,
    "5Y": 0.0385,
    "7Y": 0.0375,
    "10Y": 0.0370,
    "12Y": 0.0372,
    "15Y": 0.0375,
    "20Y": 0.0380,
    "30Y": 0.0365,
}

In [ ]:
def setup_curve_and_fitter(quotes_dict, degree=2, is_local=False, name="sandbox"):
    # 1. Tenors
    tenors = [float(t.replace("Y", "")) for t in quotes_dict.keys()]

    # 2. Quotes
    quotes = [
        SwapQuote(symbol=f"QUOTE_{t}Y", tenor=float(t), rate=quotes_dict[f"{t}Y"])
        for t in [str(int(x)) for x in tenors]
    ]

    # 3. Curve & Points
    points = [
        YieldCurvePoint(name=f"{name}_PT_{t}Y", tenor_years=t, is_fitted=True)
        for t in tenors
    ]

    if degree == 1:
        curve = LinearTermDiscountCurve(name=name, points=points)
    else:
        curve = IntegratedShortRateCurve(
            name=name, 
            currency="USD", 
            points=points, 
            degree=degree, 
            is_local=is_local
        )

    # Link quotes to points
    for pt, q in zip(points, quotes):
        pt.quote_ref = q

    # 4. Target Swaps
    target_swaps = [
        IRSwapFixedFloatApprox(
            symbol=f"SWAP_{int(q.tenor)}Y",
            tenor_years=q.tenor,
            fixed_rate=q.rate,
            curve=curve,
            notional=10e6
        ) for q in quotes
    ]

    # 5. Fitter
    fitter = CurveFitter(
        name=f"FITTER_{name}",
        curve=curve,
        points=points,
        quotes=quotes,
        target_swaps=target_swaps
    )

    return fitter, curve, target_swaps

### ⚙️ Solver Configuration
Choose your algorithm parameters here.

In [ ]:
# Configuration parameters
degree = 2      # 1: Linear, 2: Quadratic, 3: Cubic
is_local = False # True for Bessel local cubic, False for global smooth

fitter, curve, swaps = setup_curve_and_fitter(market_quotes, degree=degree, is_local=is_local)

print(f"Solving curve with degree={degree}, is_local={is_local}...")
start = time.time()
fitter.solve()
solve_time_ms = (time.time() - start) * 1000
print(f"Solve complete in {solve_time_ms:.2f}ms")

In [ ]:
# Snapshot plotting data
plot_tenors = np.linspace(0.01, 31.0, 500)
period = 1.0/365.0 # Daily forward

fwds = curve.fwd_array(plot_tenors.tolist(), period=period)
zeros = [curve.df_at(t)**(-1.0/t)-1.0 if t > 0 else curve.df_at(0.01)**(-1.0/0.01)-1.0 for t in plot_tenors]

# Format results
pillar_data = pd.DataFrame([
    {"Tenor": p.tenor_years, "Market Rate (%)": p.quote_ref.rate*100, "Fitted Zero (%)": p.fitted_rate*100}
    for p in curve._sorted_points()
])

In [ ]:
fig = go.Figure()

# Forward Rate
fig.add_trace(go.Scatter(
    x=plot_tenors, 
    y=np.array(fwds) * 100, 
    name="Inst. Forward Rate (%)",
    line=dict(color="#8b5cf6", width=2.5)
))

# Zero Rate
fig.add_trace(go.Scatter(
    x=plot_tenors, 
    y=np.array(zeros) * 100, 
    name="Zero Rate (%)",
    line=dict(color="#10b981", width=1.5, dash="dash")
))

# Pillars
fig.add_trace(go.Scatter(
    x=pillar_data["Tenor"],
    y=pillar_data["Market Rate (%)"],
    mode="markers",
    name="Market Par Rates",
    marker=dict(size=10, color="#ef4444", symbol="diamond")
))

fig.update_layout(
    title=f"Fitted Curve Analytics (Solved in {solve_time_ms:.2f}ms)",
    xaxis_title="Tenor (Years)",
    yaxis_title="Rate (%)",
    template="plotly_dark",
    height=600,
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
)

fig.show()

In [ ]:
# Residuals and Pillar Stats
residual_data = []
for s in swaps:
    residual_data.append({
        "Swap": s.symbol,
        "Tenor": s.tenor_years,
        "NPV": s.npv,
        "DV01": s.dv01,
    })

res_df = pd.DataFrame(residual_data)

print("--- Pillar Results ---")
display(pillar_data)
print("\n--- Solver Residuals (NPV) ---")
display(res_df)

### 🧪 Sensitivity Testing
Manually tweak the fitted rates below to see how the curve shape changes without re-solving the entire system.

In [ ]:
# Manual Pillar Overrides (in %)
manual_fitted_rates = {
    "1Y": pillar_data.iloc[0]['Fitted Zero (%)'],
    "2Y": pillar_data.iloc[1]['Fitted Zero (%)'],
    "3Y": pillar_data.iloc[2]['Fitted Zero (%)'],
    "5Y": pillar_data.iloc[3]['Fitted Zero (%)'],
    "7Y": pillar_data.iloc[4]['Fitted Zero (%)'],
    "10Y": pillar_data.iloc[5]['Fitted Zero (%)'],
    "12Y": pillar_data.iloc[6]['Fitted Zero (%)'],
    "15Y": pillar_data.iloc[7]['Fitted Zero (%)'],
    "20Y": pillar_data.iloc[8]['Fitted Zero (%)'],
    "30Y": pillar_data.iloc[9]['Fitted Zero (%)'],
}

# Apply tweaks here if desired, e.g.:
# manual_fitted_rates["10Y"] += 0.1 

for i, p in enumerate(curve._sorted_points()):
    tenor_str = f"{int(p.tenor_years)}Y"
    if tenor_str in manual_fitted_rates:
        p.fitted_rate = manual_fitted_rates[tenor_str] / 100.0

curve.invalidate_caches()

m_fwds = curve.fwd_array(plot_tenors.tolist(), period=1.0/365.0)

m_fig = go.Figure()
m_fig.add_trace(go.Scatter(x=plot_tenors, y=np.array(m_fwds)*100, name="Manual Forward", line=dict(color="#ec4899", width=3)))
m_fig.add_trace(go.Scatter(x=plot_tenors, y=np.array(fwds)*100, name="Last Solved Forward", line=dict(color="#8b5cf6", width=1, dash="dot")))

m_fig.update_layout(
    title="Manual Forward Rate Tweak (Live Update Simulation)",
    xaxis_title="Tenor (Years)",
    yaxis_title="Rate (%)",
    template="plotly_dark",
    height=400
)
m_fig.show()